<a href="https://colab.research.google.com/github/tiendrebeogoarlette97-bit/NLP-BERT-Sentiment-Classification/blob/main/notebooks/NLP_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification de sentiments : TF-IDF et BERT

**Objectif :** comparer une baseline TF-IDF + régression logistique à BERT-base-uncased fine-tuné sur SST-2 (GLUE). Les modèles seront évalués sur la même partition officielle de validation, avec accuracy, F1 et matrices de confusion. Les labels du test officiel ne sont pas publics. Aucune performance n'est présupposée.

## 1. Installation et configuration

Activer un GPU dans Colab : Exécution > Modifier le type d'exécution > T4 GPU. La seed fixe les générateurs aléatoires, sans garantir une reproductibilité bit à bit sur GPU.

In [ ]:
%pip install -q datasets==3.6.0 transformers==4.51.3 accelerate==1.6.0

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from transformers import set_seed
from IPython.display import display

SEED = 42
set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch :', torch.__version__)
print('Device :', DEVICE)
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
else:
    print('Attention : activer un GPU avant le fine-tuning.')

## 2. Chargement de SST-2

Nous conservons les partitions officielles. Le train sert à apprendre les paramètres et le vocabulaire TF-IDF ; la validation sert à comparer les modèles. Le test est inspecté uniquement pour sa structure.

In [ ]:
dataset = load_dataset('nyu-mll/glue', 'sst2')
print(dataset)
train_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
label_names = {0: 'Negative', 1: 'Positive'}
display(train_df.head(10))
print('Labels du test :', test_df['label'].unique())

## 3. Analyse exploratoire des données

Nous vérifions la qualité des textes, les labels, les doublons et les longueurs. Les doublons sont recensés sans suppression automatique afin de préserver le protocole officiel ; les éventuels chevauchements train/validation seront signalés comme limite.

In [ ]:
summary = []
for name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    normalized = df['sentence'].fillna('').str.strip().str.lower()
    summary.append({'split': name, 'rows': len(df), 'missing_text': df['sentence'].isna().sum(), 'empty_text': normalized.eq('').sum(), 'duplicate_texts': normalized.duplicated().sum(), 'label_values': sorted(df['label'].unique().tolist())})
display(pd.DataFrame(summary))
assert train_df['label'].isin([0, 1]).all()
assert val_df['label'].isin([0, 1]).all()
train_texts = set(train_df['sentence'].str.strip().str.lower())
val_texts = set(val_df['sentence'].str.strip().str.lower())
print('Textes distincts communs train/validation :', len(train_texts & val_texts))

In [ ]:
eda = pd.concat([train_df.assign(split='train'), val_df.assign(split='validation')], ignore_index=True)
eda['sentiment'] = eda['label'].map(label_names)
eda['word_count'] = eda['sentence'].str.split().str.len()
counts = pd.crosstab(eda['split'], eda['sentiment'])
display(counts)
display(pd.crosstab(eda['split'], eda['sentiment'], normalize='index').mul(100).round(2))
display(eda.groupby('split')['word_count'].describe(percentiles=[.5, .9, .95, .99]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, split in zip(axes, ['train', 'validation']):
    sns.countplot(data=eda[eda['split'] == split], x='sentiment', order=['Negative', 'Positive'], ax=ax)
    ax.set_title(f'Distribution des classes — {split}')
    ax.set_xlabel('Sentiment')
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(data=eda, x='word_count', hue='split', stat='density', common_norm=False, element='step', bins=40)
plt.xlabel('Nombre de mots par phrase')
plt.title('Distribution des longueurs de phrases')
plt.tight_layout()
plt.show()

### Interprétation à compléter après exécution

Examiner l'équilibre des classes, les valeurs manquantes, les doublons et les longueurs. Le nombre de mots n'est pas le nombre de tokens WordPiece : la longueur maximale BERT sera vérifiée lors de la tokenisation.